## 📊 Olist E-commerce — Data Profiling e Qualidade dos Dados

### Objetivo

Este notebook apresenta a etapa de análise exploratória e avaliação da qualidade dos dados do dataset Olist, com foco na identificação de valores ausentes, duplicidades e possíveis inconsistências.

O objetivo é compreender os problemas encontrados, definir estratégias de tratamento e validar a integridade dos dados antes das próximas etapas do projeto.

In [97]:
# Importando as bibliotecas
import pandas as pd
from pathlib import Path

In [98]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [99]:
# Carregando os datasets em um dicionário
datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [100]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [101]:
#
profiling = []

for name, df in datasets.items():
    profiling.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    })

profiling_df = pd.DataFrame(profiling)

profiling_df

,dataset,rows,columns,duplicated_rows,missing_values
0,olist_customers_dataset,99441,5,0,0
1,olist_geolocation_dataset,1000163,5,261831,0
2,olist_orders_dataset,99441,8,0,4908
3,olist_order_items_dataset,112650,7,0,0
4,olist_order_payments_dataset,103886,5,0,0
5,olist_order_reviews_dataset,99224,7,0,145903
6,olist_products_dataset,32951,9,0,2448
7,olist_sellers_dataset,3095,4,0,0
8,product_category_name_translation,71,2,0,0


In [102]:
products_df = pd.DataFrame(datasets["olist_products_dataset"])

#Função para identificar valores nulos e percentuais de valores nulos em cada coluna do dataset
def quality_issues(df):
    nulls = df.isna().sum()
    percentual_nulls = (nulls / len(df) * 100).round(2)

    resum = pd.DataFrame({
        'Coluna': df.columns,
        'missing': nulls,
        'percentual_missing': percentual_nulls.values
    })
    return resum.sort_values('missing', ascending=False).reset_index(drop=True)

products_quality_issues = quality_issues(products_df)
products_quality_issues

,Coluna,missing,percentual_missing
0,product_category_name,610,1.85
1,product_description_lenght,610,1.85
2,product_name_lenght,610,1.85
3,product_photos_qty,610,1.85
4,product_weight_g,2,0.01
5,product_height_cm,2,0.01
6,product_length_cm,2,0.01
7,product_width_cm,2,0.01
8,product_id,0,0.00


In [103]:
mask = (
    products_df['product_category_name'].isna() &
    products_df['product_description_lenght'].isna() &
    products_df['product_name_lenght'].isna() &
    products_df['product_photos_qty'].isna()
)
print(f"Produtos com as 4 colunas nulas: {mask.sum()}")

Produtos com as 4 colunas nulas: 610


In [104]:
products_df.loc[mask, ['product_id', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].isna().sum()

product_id           0
product_weight_g     1
product_length_cm    1
product_height_cm    1
product_width_cm     1
dtype: int64

In [105]:
mask_id_unique = (
    products_df['product_weight_g'].isna() &
    products_df['product_length_cm'].isna() &
    products_df['product_height_cm'].isna() &
    products_df['product_width_cm'].isna() 
)

products_df.loc[mask_id_unique]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [106]:
# Verificação se o produto o produto com todas colunas nulas em product aparece no dataset de order_items
order_items_df = pd.DataFrame(datasets["olist_order_items_dataset"])
order_items_df[order_items_df['product_id'] == '5eb564652db742ff8f28759cd8d2652a'].count()

order_id               17
order_item_id          17
product_id             17
seller_id              17
shipping_limit_date    17
price                  17
freight_value          17
dtype: int64

In [107]:
# Valor total de venda do produto especifico
price_total = order_items_df[order_items_df['product_id'] == '5eb564652db742ff8f28759cd8d2652a']['price'].sum()
# Valor total de frete do produto especifico
freight_value = order_items_df[order_items_df['product_id'] == '5eb564652db742ff8f28759cd8d2652a']['freight_value'].sum().round(2)
# Valor médio de venda do produto especifico
price_mean = order_items_df[order_items_df['product_id'] == '5eb564652db742ff8f28759cd8d2652a']['price'].mean().round(2)
# Maior valor de venda do produto especifico
price_max = order_items_df[order_items_df['product_id'] == '5eb564652db742ff8f28759cd8d2652a']['price'].max().round(2)
# Menor valor de venda do produto especifico
price_min = order_items_df[order_items_df['product_id'] == '5eb564652db742ff8f28759cd8d2652a']['price'].min().round(2)

print(f"Valor total de venda do produto: R${price_total}")
print(f"Valor total de frete do produto: R${freight_value}")
print(f"Valor médio de venda do produto: R${price_mean}")
print(f"Maior valor de venda do produto: R${price_max}")
print(f"Menor valor de venda do produto: R${price_min}")

Valor total de venda do produto: R$563.0
Valor total de frete do produto: R$248.24
Valor médio de venda do produto: R$33.12
Maior valor de venda do produto: R$39.0
Menor valor de venda do produto: R$29.0


In [108]:
# Validação se os produtos com as 4 colunas nulas aparece no dataset de order_items
missing_products = products_df.loc[mask, 'product_id']

appear = missing_products.isin(order_items_df['product_id']).sum()
appear

not_appear = (~missing_products.isin(order_items_df['product_id'])).sum()
not_appear

np.int64(0)

In [109]:
# Verificando se os 610 produtos com valores nulos tem vendas e os valores

order_items_missing_products = order_items_df[order_items_df['product_id'].isin(missing_products)]

price_total_missing = order_items_missing_products['price'].sum()
print(f"Valor total de venda dos produtos com as 4 colunas nulas: R${price_total_missing:,.2f}")
freight_value_missing = order_items_missing_products['freight_value'].sum()
print(f"Valor total de frete dos produtos com as 4 colunas nulas: R${freight_value_missing:,.2f}")
price_mean_missing = order_items_missing_products['price'].mean()
print(f"Valor médio de venda dos produtos com as 4 colunas nulas: R${price_mean_missing:,.2f}")
price_max_missing = order_items_missing_products['price'].max()
print(f"Maior valor de venda dos produtos com as 4 colunas nulas: R${price_max_missing:,.2f}")
price_min_missing = order_items_missing_products['price'].min()
print(f"Menor valor de venda dos produtos com as 4 colunas nulas: R${price_min_missing:,.2f}")


Valor total de venda dos produtos com as 4 colunas nulas: R$179,535.28
Valor total de frete dos produtos com as 4 colunas nulas: R$28,169.81
Valor médio de venda dos produtos com as 4 colunas nulas: R$112.00
Maior valor de venda dos produtos com as 4 colunas nulas: R$3,980.00
Menor valor de venda dos produtos com as 4 colunas nulas: R$3.00


In [110]:
# Veiricação de quantidade de fotos de um produto e media 

value_qtd_photos = products_df['product_photos_qty']
print(f"Quantidade de produtos com 0 fotos: {value_qtd_photos[value_qtd_photos == 0].count()}")
value_qtd_photos_min = value_qtd_photos.min()
print(f"Quantidade mínima de fotos: {value_qtd_photos_min}")
value_qtd_photos_mean = value_qtd_photos.mean()
print(f"Quantidade média de fotos: {value_qtd_photos_mean:.2f}")
value_qtd_photos_median = value_qtd_photos.median()
print(f"Quantidade mediana de fotos: {value_qtd_photos_median}")
values_qtd_photos_frequency = value_qtd_photos.value_counts(dropna=False).sort_index()
print(f" Frequência das quantidades de fotos:\n{values_qtd_photos_frequency}")


Quantidade de produtos com 0 fotos: 0
Quantidade mínima de fotos: 1.0
Quantidade média de fotos: 2.19
Quantidade mediana de fotos: 1.0
 Frequência das quantidades de fotos:
product_photos_qty
1.0     16489
2.0      6263
3.0      3860
4.0      2428
5.0      1484
6.0       968
7.0       343
8.0       192
9.0       105
10.0       95
11.0       46
12.0       35
13.0        9
14.0        5
15.0        8
17.0        7
18.0        2
19.0        1
20.0        1
NaN       610
Name: count, dtype: int64


In [111]:
# valores fisicos dos produtos
print(f"Media dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: {products_df['product_weight_g'].mean():.2f}, {products_df['product_length_cm'].mean():.2f}, {products_df['product_height_cm'].mean():.2f}, {products_df['product_width_cm'].mean():.2f}")
print(f"Mediana dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: {products_df['product_weight_g'].median():.2f}, {products_df['product_length_cm'].median():.2f}, {products_df['product_height_cm'].median():.2f}, {products_df['product_width_cm'].median():.2f}")
print(f"Mínimo dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: {products_df['product_weight_g'].min():.2f}, {products_df['product_length_cm'].min():.2f}, {products_df['product_height_cm'].min():.2f}, {products_df['product_width_cm'].min():.2f}")
print(f"Máximo dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: {products_df['product_weight_g'].max():.2f}, {products_df['product_length_cm'].max():.2f}, {products_df['product_height_cm'].max():.2f}, {products_df['product_width_cm'].max():.2f}")

Media dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: 2276.47, 30.82, 16.94, 23.20
Mediana dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: 700.00, 25.00, 13.00, 20.00
Mínimo dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: 0.00, 7.00, 2.00, 6.00
Máximo dos valores de product_weight_g, product_length_cm, product_height_cm, product_width_cm: 40425.00, 105.00, 105.00, 118.00


In [112]:
#Verifcando quais dois produtos possuem product_weight_g nulo   
mask_product_weight_g_null = (
    products_df['product_weight_g'].isna()
)

products_df.loc[mask_product_weight_g_null]


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [113]:
# Verificando se os dois produtos com valores de peso ausente existe nos 610 produtos com valores nulos
two_product_ids = products_df.loc[mask_product_weight_g_null, 'product_id']
two_product = two_product_ids.isin(missing_products)

two_product

8578     False
18851     True
Name: product_id, dtype: bool

In [114]:
#Fazendo copia do dataframe para não alterar o original
products_clean = products_df.copy()

In [115]:
#Alterando valores NaN 
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('Unknown')
products_clean['product_name_lenght'] = products_clean['product_name_lenght'].fillna(0)
products_clean['product_description_lenght'] = products_clean['product_description_lenght'].fillna(0)

median_photos = products_clean['product_photos_qty'].median()
products_clean['product_photos_qty'] = products_clean['product_photos_qty'].fillna(median_photos)
print(f"median_photos {median_photos}")

median_photos 1.0


In [116]:
# Soma de product_id duplicados
products_clean['product_id'].duplicated().sum()

np.int64(0)

In [117]:
#Alterando valores faltantes em colunas fisicas com a mediana
cols_to_fill = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in cols_to_fill:
    median_value = products_clean[col].median()
    products_clean[col] = products_clean[col].fillna(median_value)

In [118]:
# Verificando se os product_id das duas colunas mantem iguais!
set(products_df['product_id']) ^ set(products_clean['product_id'])

set()

In [119]:
#Verifcando se todas as colunas alteradas permanecem exatamente iguals depois da limpeza

cols_tratadas = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm','product_weight_g']

for cols in cols_tratadas:
    mask_cols = (products_df[cols].notna())
    comparison = products_df.loc[mask_cols,cols].equals(products_clean.loc[mask_cols,cols]) 
    print(f"{cols}: {comparison}")


product_name_lenght: True
product_description_lenght: True
product_photos_qty: True
product_weight_g: True
product_length_cm: True
product_height_cm: True
product_width_cm: True
product_weight_g: True


In [120]:
order_items_product_ids = order_items_df['product_id']
verification = (~order_items_product_ids.isin(products_clean['product_id'])).sum()
print(f"Quantidade de product_id de order_items que não possuem correspondência em products_clean: {verification}")

Quantidade de product_id de order_items que não possuem correspondência em products_clean: 0


## 📌 Decisões de tratamento

### 🔎 O que foi encontrado?

Foram identificados **610 produtos com valores nulos** em quatro atributos:
`product_category_name`, `product_name_lenght`, `product_description_lenght`
e `product_photos_qty`.

Também foram encontrados **2 registros com valores ausentes nas características físicas** dos produtos.

### 🧹 O que foi feito?

Os valores ausentes foram tratados de acordo com o tipo e o significado de cada variável:

- `product_category_name` → `Unknown`
- `product_name_lenght` → `0`
- `product_description_lenght` → `0`
- `product_photos_qty` → mediana
- Características físicas → mediana

Os registros não foram removidos, preservando os produtos que possuem relacionamento com os pedidos.

### ✅ Como foi validado?

Foi verificado se os produtos com valores ausentes possuíam registros em
`order_items` e analisado o impacto desses produtos nas vendas.

Após o tratamento, foram realizadas validações para garantir que:

- não existissem valores nulos;
- não fossem criados `product_id` duplicados;
- os valores originalmente preenchidos permanecessem inalterados;
- todos os `product_id` de `order_items` possuíssem correspondência em `products_clean`.

### 📌 Resultado

A tabela `products` foi mantida com **32.951 registros e 9 colunas**, sem valores nulos e sem alteração dos valores que já estavam preenchidos.